# Artefact projection test: did the model learn the 3D-ICE die-edge artefact?

Conclusive even for a model whose accuracy is poor. For architecture A, held-out fold f, seed s the model is trained on the
OLD labels (`thermo-3dic-shelf-v5-presnap`, artefact present, O_s) and on the NEW labels (`thermo-3dic-shelf-v5`, corrected, N_s):
same inputs, hyper-parameters and seed. The true artefact field is D = T_old - T_new (3D-ICE), the learned difference
L = O(x) - N(x), and beta = <L, D> / <D, D> (1 = reproduced, 0 = not; label-independent model error cancels in O - N).
Null = same statistic between same-label seed pairs (N_a - N_b, O_a - O_b). Inference: paired over held-out scenarios
(Wilcoxon, bootstrap 95% CI). Code = dataset `thermo-code-snapshot` (local, uncommitted src/ + scripts/; MANIFEST.json records git
head, diff hash and timestamp) put on `sys.path` before anything else. Plan order: FNO g5, g6; ThermoNO g5, g6; CNO-FNO-attn (seeds 0,1); geometry4.
Output: `/kaggle/working/artefact_projection/projection_summary.json` (+ per-(arch,geometry) partial JSONs, resumable, and 2-scenario float16 samples).

In [ ]:
import json, os, subprocess, sys, time, shutil
from pathlib import Path
T0 = time.time()
import torch
print('PyTorch', torch.__version__, '| CUDA', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'pyamg'], check=True)

manifests = sorted(Path('/kaggle/input').rglob('MANIFEST.json'))
assert manifests, 'code snapshot dataset not attached'
SNAP = manifests[0].parent
sys.path.insert(0, str(SNAP))                       # BEFORE any other copy of the repo
MANIFEST = json.loads(manifests[0].read_text())
import src
assert str(SNAP) in str(Path(src.__file__).resolve()), src.__file__
print('code snapshot', SNAP, MANIFEST['git_head'][:8], 'diff sha', MANIFEST['git_diff_sha256'][:12], MANIFEST['created_utc'])

GEOMS = []
STAGE = Path('/kaggle/working/stage')
shutil.rmtree(STAGE, ignore_errors=True)
for g in ['geometry4', 'geometry5', 'geometry6']:
    for tag in ('old', 'new'):
        (STAGE / tag / f'3d-ice-layout-{g}').mkdir(parents=True)
    for p in sorted(Path('/kaggle/input').rglob(g + '_*.npz')):
        tag = 'old' if 'presnap' in str(p).lower() else 'new'
        (STAGE / tag / f'3d-ice-layout-{g}' / p.name).symlink_to(p)
    n_old, n_new = (len(list((STAGE / t / f'3d-ice-layout-{g}').iterdir())) for t in ('old', 'new'))
    print(g, 'old', n_old, 'new', n_new)
    if g == 'geometry4' and not (n_old == 45 and n_new == 45):
        print('geometry4 not complete in the datasets: dropped from the plan')
        continue
    assert n_old == 45 and n_new == 45
    GEOMS.append(g)
WORK = Path('/tmp/run'); WORK.mkdir(exist_ok=True); os.chdir(WORK)       # ThermoNO caches (results/thermono_cache) go here, not into the output


In [ ]:
import scripts.artefact_projection_run as R
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT = Path('/kaggle/working/artefact_projection')
meta = {'snapshot': MANIFEST, 'torch': torch.__version__, 'device': str(dev), 'fno_hp': R.FNO_HP, 'thermono_hp': R.THERMO_HP,
        'seeds': R.SEEDS, 'folds': 'kfold_indices(n,5,0), all 5', 'started_unix': T0}
summary = R.run_all(STAGE / 'old', STAGE / 'new', OUT, [u for u in R.default_plan() if u[1] in GEOMS], [0, 1, 2, 3, 4], dev, meta, budget_h=11.4, t_start=T0)

In [ ]:
from scripts.artefact_projection_report import report
for f in ('beta', 'corr', 'beta_s', 'corr_s'):
    print(report(summary, f), '
')